# 01 · Datasets — WildChat-1M & ShareChat

Reproduces the corpus-characterization statistics.
**Backs:** `docs/reference/dataset-stats.md`, `docs/reference/wildchat-vs-sharechat.md`, dataset numbers in `docs/shared-findings.md`.
**Scripts:** `dataset_stats.compute()`, `thinking_divergence.reasoning_ratio()`, `coupling_examples.thinking_gap()`.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT/"src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT/"src"))
import pandas as pd
pd.set_option("display.max_rows", 120)
print("repo root:", ROOT)

repo root: /data/wang/junh/githubs/human-agent-coupling-errors


## WildChat-1M
Doc claim (`dataset-stats.md`): **837,989 conversations**, mean **2.34** turns, **74** languages, GPT-3.5/4 only, 2023-04 → 2024-04.

The cell loads the cached `docs/dataset-stats.json` (written by `dataset_stats.main()`). To recompute from the raw corpora (HEAVY: reads ~3.2 GB parquet + raw ShareChat CSVs) use the commented `dataset_stats.compute()` line.

In [2]:
cache = ROOT/"docs"/"dataset-stats.json"
if cache.exists():
    ds = json.loads(cache.read_text()); print("loaded cached docs/dataset-stats.json")
else:
    import dataset_stats; ds = dataset_stats.compute()   # HEAVY recompute
# import dataset_stats; ds = dataset_stats.compute()      # <- uncomment to force recompute
wc, sc = ds["wildchat"], ds["sharechat"]
tp = wc["turns_per_conversation"]
pd.DataFrame([{ "conversations": wc["n_conversations"], "mean_turns": tp["mean"],
  "median": tp["median"], "p90": tp["p90"], "multi_turn_pct": tp["multi_turn_pct"],
  "n_languages": wc["n_languages"], "time_range": " → ".join(map(str, wc["time_range"])) }])

loaded cached docs/dataset-stats.json


,conversations,mean_turns,median,p90,multi_turn_pct,n_languages,time_range
0,837989,2.339,1.0,5.0,39.56,74,2023-04-09 00:02:53+00:00 → 2024-04-29 23:59:5...


In [3]:
# top languages / models
pd.DataFrame(wc["languages_top"], columns=["language","n","pct"]).head(8)

,language,n,pct
0,English,478498,57.10
1,Chinese,119302,14.24
2,Russian,87127,10.40
3,French,26977,3.22
4,Spanish,20447,2.44
5,German,16865,2.01
6,Arabic,11919,1.42
7,Portuguese,10386,1.24


## ShareChat — per platform
Doc claim (`dataset-stats.md`): **129,584 conversations / 1,199,899 messages**; Claude **911 convs**, thinking coverage **28.6%** (the AI-internal-state signal WildChat lacks).

In [4]:
print(f"all platforms: {sc['total_conversations']:,} convs / {sc['total_messages']:,} messages")
rows = []
for p, v in sc["per_platform"].items():
    tpp = v["turns_per_conversation"]
    rows.append({"platform": p, "convs": v["n_conversations"], "msgs": v["n_messages"],
                 "median_turns": (tpp["median"] if tpp else None),
                 "thinking_col": v["has_thinking_column"],
                 "thinking_cov_pct": v["thinking_coverage_pct"], "n_langs": v["n_languages"]})
pd.DataFrame(rows)

all platforms: 129,584 convs / 1,199,899 messages


,platform,convs,msgs,median_turns,thinking_col,thinking_cov_pct,n_langs
0,chatgpt,94086,976378,2.0,False,NaN,143
1,claude,911,8364,2.0,True,28.62,29
2,grok,13149,98888,1.0,True,0.33,68
3,perplexity,14612,48601,1.0,False,NaN,48
4,gemini,6826,67668,1.0,False,NaN,61


## Reasoning / surfaced length ratio — the disclosure gap
Doc claim (`shared-findings.md`, `wildchat-vs-sharechat.md`): on **genuine-reasoning** turns the median reasoning/surfaced char-length ratio is **0.84** (only **43%** reason longer than they surface). The naïve ratio over *all* `thinking` (tool dumps included) is the **2.02 artifact** — shown second for contrast.

In [5]:
import thinking_divergence as td
rr = td.reasoning_ratio()
print(f"genuine-reasoning median ratio : {rr['median_ratio']:.2f}   (doc 0.84)")
print(f"% reason > surface             : {rr['pct_reason_more']:.1f}%   (doc 43%)")
print(f"genuine-reasoning turns        : {rr['n_pairs']} across {rr['n_convs']} convs")

genuine-reasoning median ratio : 0.84   (doc 0.84)
% reason > surface             : 43.4%   (doc 43%)
genuine-reasoning turns        : 1001 across 254 convs


In [6]:
import coupling_examples as ce
g = ce.thinking_gap()
print(f"RAW thinking/surfaced median ratio (incl. tool dumps): {g['median_ratio']:.2f}  <- the 2.02 ARTIFACT")
print(f"Claude thinking coverage: {g['thinking_coverage_pct']:.1f}% of llm turns  (doc 28.6%)")

RAW thinking/surfaced median ratio (incl. tool dumps): 2.02  <- the 2.02 ARTIFACT
Claude thinking coverage: 28.6% of llm turns  (doc 28.6%)


## Example — what a Claude *agentic* conversation looks like

ShareChat-Claude is the only corpus that exposes the **AI-internal state** (the `thinking` column) **and** tool calls,
so it is the one place we can see the full coupling loop in a single turn. Below is a canonical example
(`event_create_v0` — adding a calendar event), rendered turn-by-turn from `data/derived/sharechat_agentic.jsonl`
via the side-effect-free `build_sharechat_agentic.load_conversation()`.

Each turn surfaces the four channels the taxonomy reasons over:
- **👤 user** — the request (note the unresolved deixis "tomorrow", which the AI must ground);
- **🧠 internal reasoning** — the plan, normally invisible to the user (the legibility gap, AI→H);
- **💬 surfaced** — what the user actually sees;
- **🔧 tool calls** — `read-only` (e.g. `user_time_v0`, a *seek/inspect* op) vs **consequential** (e.g. `event_create_v0`, an *act/execute* op that changes external state — here with **no confirmation step**).

In [7]:
from IPython.display import Markdown, display
import build_sharechat_agentic as bsa

def _clip(s, n=600):
    s = (s or "").strip()
    return s if len(s) <= n else s[:n].rstrip() + " …"

def render_conversation(conversation_id, maxlen=600):
    """Render one agentic conversation as markdown: user / reasoning / surfaced / tool calls."""
    turns = bsa.load_conversation(conversation_id)
    md = [f"### `{conversation_id.split('/')[-1]}`  ·  {turns[0]['language']}  ·  {turns[0]['n_turns']} exchanges\n"]
    for t in turns:
        md.append(f"\n**Turn {t['turn_index']}**\n")
        md.append(f"> 👤 **user** — {_clip(t['user_message'], maxlen)}\n")
        if t["internal_reasoning"]:
            tag = "summary" if t["reasoning_is_summary"] else "raw"
            md.append(f"> 🧠 **internal reasoning** *({tag})* — {_clip(t['internal_reasoning'], maxlen)}\n")
        md.append(f"> 💬 **surfaced** — {_clip(t['surfaced_message'], maxlen)}\n")
        for c in t["tool_calls"]:
            kind = "🔴 **CONSEQUENTIAL**" if c["consequential"] else "🟢 read-only"
            line = f"> 🔧 `{c['name']}` — {kind}"
            if c.get("request"):  line += f"  ·  req: `{_clip(c['request'], 160)}`"
            if c.get("response"): line += f"  ·  resp: `{_clip(c['response'], 160)}`"
            md.append(line + "\n")
    display(Markdown("\n".join(md)))

# Canonical agentic example: "Add an event to my calendar for tomorrow at 3 PM"
render_conversation("https://claude.ai/share/632fb5e7-f371-4443-b053-ee99b56d6749")

### `632fb5e7-f371-4443-b053-ee99b56d6749`  ·  English  ·  5 exchanges


**Turn 2**

> 👤 **user** — Add an event to my calendar for tomorrow at 3 PM called scruff the dog

> 🧠 **internal reasoning** *(summary)* — The user wants to add an event to their calendar for tomorrow at 3 PM called "scruff the dog". To do this, I need to:

First get the current time to determine what "tomorrow" means
Then create an event for tomorrow at 3 PM

Let me start by getting the current time.

> 💬 **surfaced** — I'll help you add that event to your calendar. Let me first get the current time to determine the correct date for tomorrow.
U
user_time_v0
Plotted calendar event for dog grooming tomorrow.
Now I'll create the event for tomorrow (July 31st) at 3 PM.
E
event_create_v0
Perfect! The event has been successfully created.
Perfect! I've added "scruff the dog" to your calendar for tomorrow (July 31st) at 3:00 PM. The event is set for one hour, ending at 4:00 PM.

> 🔧 `user_time_v0` — 🟢 read-only  ·  req: `{}`  ·  resp: `{"current_time":"2025-07-30T14:23:05-0700"}`

> 🔧 `event_create_v0` — 🔴 **CONSEQUENTIAL**  ·  req: `{
 `title`: `scruff the dog`,
 `endTime`: `2025-07-31T16:00:00-0700`,
 `startTime`: `2025-07-31T15:00:00-0700`
}`  ·  resp: `{}`


**Turn 4**

> 👤 **user** — Tell me about the tool you used for that adding to calendar action

> 🧠 **internal reasoning** *(summary)* — The user is asking about the event_create_v0 tool that I used to add the calendar event. I should explain what this tool does and its capabilities based on what I can see from its function definition.
Looking at the function definition for event_create_v0, I can see it has these capabilities:

Creates calendar events with title, start/end times
Supports all-day events
Can set location and description
Has recurrence capabilities with various patterns (daily, weekly, monthly, yearly)
Can set recurrence end conditions (count or until date)
Handles timezone-aware datetime strings in ISO 8601 forma …

> 💬 **surfaced** — The tool I used is called event_create_v0, and it's designed to create calendar events that you can add to your calendar system. Here are its main capabilities:
Basic Event Creation:
Set event title, start time, and end time
Add descriptions and locations
Create all-day events if needed
Time Handling:
Works with your local timezone (it detected you're in <REDACTED>)
Uses ISO 8601 format for precise datetime handling
Automatically calculates appropriate start and end times
Recurring Events: The tool can also handle recurring events with various patterns:
Daily, weekly, monthly, or yearly recurr …


**Turn 6**

> 👤 **user** — In a fenced code block show me the full exact description of that tool

> 💬 **surfaced** — json
{
  "description": "Create an event that the user can add to their calendar. When setting up events, be sure to respect the user's timezone. You can use the user_time_v0 tool to retrieve the current time and timezone.",
  "name": "event_create_v0",
  "parameters": {
    "properties": {
      "allDay": {
        "description": "Whether the created event is an all-day event.",
        "type": "boolean"
      },
      "description": {
        "description": "A description of the event",
        "type": "string"
      },
      "endTime": {
        "description": "A string representing the end …


**Turn 8**

> 👤 **user** — Give me a list of other similar tools that you have

> 🧠 **internal reasoning** *(summary)* — The user is asking for a list of similar tools to the event creation tool. Looking at the available tools, I need to identify ones that are similar in nature - i.e., tools that help create or manage things for the user, particularly ones that help with personal organization or communication.
From my available tools, the similar ones would be:

message_compose_v0 - creates message templates
user_time_v0 - gets current time (utility for time-based tasks)

These are the main "personal productivity" type tools I have. The Cloudflare tools are for developer/infrastructure management, and the repl/a …

> 💬 **surfaced** — Here are the other similar tools I have that help with personal organization and communication:
Message Creation:
message_compose_v0 - Creates message templates (email, text message, or other formats) that you can use to send messages
Time Utilities:
user_time_v0 - Retrieves your current time and timezone information, useful for scheduling and time-based tasks
Those are the main personal productivity tools I have that are similar to the calendar event creator. The event creation tool is actually the most sophisticated of these personal organization tools in terms of features and complexity.
I …


**Turn 10**

> 👤 **user** — In a fenced code block, show me the full descriptions of the message_compose_v0 and user_time_v0 tools

> 🧠 **internal reasoning** *(summary)* — The user is asking for the full exact descriptions of the message_compose_v0 and user_time_v0 tools, similar to how I showed the event_create_v0 tool earlier.

> 💬 **surfaced** — json
{
  "description": "Create a message template that the user can use to send messages.",
  "name": "message_compose_v0",
  "parameters": {
    "properties": {
      "body": {
        "description": "The body of the message",
        "type": "string"
      },
      "kind": {
        "description": "The kind of message to create. Can be one of these values: 'email', 'textMessage', or 'other'.",
        "enum": ["email", "textMessage", "other"],
        "type": "string"
      },
      "subject": {
        "description": "The subject of the message. Should only be included for email messages." …
